In [ ]:
import os, time, json, csv
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np


ROOTS = ["Data", "New Data"]   # dataset roots to combine
SPLITS = ["train", "valid", "test"]
batch_size = 16
num_epochs = 15
learning_rate = 1e-4
num_workers = 0                # safe for Jupyter on Windows
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin_memory = torch.cuda.is_available()

model_save_path = "lung_cancer_stage_model_best_6classes.pth"
checkpoint_dir = "checkpoints_6classes"
os.makedirs(checkpoint_dir, exist_ok=True)
log_csv = "training_log_6classes.csv"

print("Device:", device, "| CUDA available:", torch.cuda.is_available())
print("Batch size:", batch_size, "Epochs:", num_epochs, "Num workers:", num_workers)


data_transforms = {
    "train": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    "valid": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
    "test": transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    ]),
}


CANONICAL_CLASSES = [
    "normal",
    "Benign cases",
    "adenocarcinoma",
    "large.cell.carcinoma",
    "Malignant cases",
    "squamous.cell.carcinoma"
]

def map_folder_to_canonical(folder_name: str) -> str | None:
    """Map arbitrary folder name to one of the 6 canonical classes, or return None."""
    n = folder_name.lower().replace("_"," ").replace(".", " ")
    # direct exact synonyms
    if "normal" in n:
        return "normal"
    if "adenocarcinoma" in n:
        return "adenocarcinoma"
    if "large cell" in n or "large.cell" in folder_name.lower() or "large" in n and "cell" in n:
        return "large.cell.carcinoma"
    if "squamous" in n:
        return "squamous.cell.carcinoma"
    # benign synonyms
    if "benign" in n:
        return "Benign cases"
    # malignant synonyms
    if "malignant" in n:
        return "Malignant cases"
    # not recognized -> ignore
    return None


class_names = CANONICAL_CLASSES.copy()
class_to_idx = {c: i for i, c in enumerate(class_names)}
print("Enforced class names (6 classes):")
for i,c in enumerate(class_names):
    print(f"  [{i}] {c}")


def collect_samples_for_split(roots, split, class_to_idx):
    samples = []
    for r in roots:
        p_split = os.path.join(r, split)
        if not os.path.isdir(p_split):
            continue
        # iterate directories under p_split and map them
        for folder in os.listdir(p_split):
            folder_path = os.path.join(p_split, folder)
            if not os.path.isdir(folder_path):
                continue
            mapped = map_folder_to_canonical(folder)
            if mapped is None:
                # skip folders that don't map to the six canonical classes
                continue
            idx = class_to_idx[mapped]
            for fname in os.listdir(folder_path):
                if fname.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff")):
                    samples.append((os.path.join(folder_path, fname), idx))
    return samples

class CombinedImageDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        try:
            img = Image.open(path).convert("RGB")
        except Exception as e:
            print("Warning - cannot open image:", path, "->", e)
            img = Image.new("RGB", (224,224), (0,0,0))
        if self.transform:
            img_t = self.transform(img)
        else:
            from torchvision.transforms import ToTensor
            img_t = ToTensor()(img)
        return img_t, label


datasets = {}
dataloaders = {}
dataset_sizes = {}

for split in SPLITS:
    samples = collect_samples_for_split(ROOTS, split, class_to_idx)
    print(f"{split} samples found: {len(samples)}")
    if len(samples) > 0:
        print("  first examples:", samples[:3])
    ds = CombinedImageDataset(samples, transform=data_transforms[split])
    shuffle = True if split == "train" else False
    loader = DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=num_workers, pin_memory=pin_memory)
    datasets[split] = ds
    dataloaders[split] = loader
    dataset_sizes[split] = len(ds)

if dataset_sizes["train"] == 0:
    raise RuntimeError("No training samples found for the six canonical classes. Check folder names and mapping rules.")


class DualResNet18(nn.Module):
    def __init__(self, num_types, num_stages=4):
        super(DualResNet18, self).__init__()
        base_model = models.resnet18(weights="IMAGENET1K_V1")
        self.backbone = nn.Sequential(*list(base_model.children())[:-1])
        num_features = base_model.fc.in_features
        self.type_head = nn.Sequential(
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_types)
        )
        self.stage_head = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_stages)
        )
    def forward(self, x):
        f = self.backbone(x)
        f = f.view(f.size(0), -1)
        return self.type_head(f), self.stage_head(f)


def get_stage_label_for_canonical(canonical_name):
    n = canonical_name.lower()
    if "normal" in n:
        return 0
    if "adenocarcinoma" in n:
        return 1
    if "large.cell" in n or "large cell" in n:
        return 2
    if "squamous" in n:
        return 3
    if "benign" in n:
        return 1
    if "malignant" in n:
        return 3
    return 2

class_idx_to_stage = {class_to_idx[c]: get_stage_label_for_canonical(c) for c in class_names}
print("Class idx -> stage mapping:")
for cname, idx in class_to_idx.items():
    print(f"  {idx}: {cname} -> Stage {class_idx_to_stage[idx]}")


num_types = len(class_names)   # should be 6
num_stages = 4
model = DualResNet18(num_types=num_types, num_stages=num_stages).to(device)

criterion_type = nn.CrossEntropyLoss()
criterion_stage = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)


best_val_acc = 0.0
history = []

# CSV header
with open(log_csv, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["epoch", "train_loss", "val_loss", "val_acc"])

for epoch in range(1, num_epochs+1):
    t0 = time.time()
    model.train()
    running_loss = 0.0
    running_samples = 0

    train_loader = dataloaders["train"]
    for inputs, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs} - Train", leave=False):
        inputs = inputs.to(device); labels = labels.to(device)
        stage_labels = torch.tensor([class_idx_to_stage[int(l.item())] for l in labels], dtype=torch.long).to(device)

        optimizer.zero_grad()
        type_out, stage_out = model(inputs)
        loss_type = criterion_type(type_out, labels)
        loss_stage = criterion_stage(stage_out, stage_labels)
        loss = loss_type + 0.5 * loss_stage
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        running_samples += inputs.size(0)

    train_loss = running_loss / (running_samples if running_samples>0 else 1)

    # Validation
    model.eval()
    val_loss = 0.0; val_samples = 0; correct = 0
    with torch.no_grad():
        for inputs, labels in tqdm(dataloaders["valid"], desc=f"Epoch {epoch}/{num_epochs} - Val", leave=False):
            inputs = inputs.to(device); labels = labels.to(device)
            stage_labels = torch.tensor([class_idx_to_stage[int(l.item())] for l in labels], dtype=torch.long).to(device)
            type_out, stage_out = model(inputs)
            loss_type = criterion_type(type_out, labels)
            loss_stage = criterion_stage(stage_out, stage_labels)
            loss = loss_type + 0.5 * loss_stage
            val_loss += loss.item() * inputs.size(0)
            preds = torch.argmax(type_out, dim=1)
            correct += torch.sum(preds == labels).item()
            val_samples += inputs.size(0)

    val_loss = val_loss / (val_samples if val_samples>0 else 1)
    val_acc = correct / (val_samples if val_samples>0 else 1)
    epoch_time = time.time() - t0

    print(f"Epoch {epoch}/{num_epochs}  time:{epoch_time:.1f}s  train_loss:{train_loss:.4f}  val_loss:{val_loss:.4f}  val_acc:{val_acc:.4f}")

    history.append({"epoch":epoch, "train_loss":train_loss, "val_loss":val_loss, "val_acc":val_acc})
    with open(log_csv, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, train_loss, val_loss, val_acc])

    ckpt_path = os.path.join(checkpoint_dir, f"ckpt_epoch{epoch}.pth")
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "class_names": class_names,
        "class_idx_to_stage": class_idx_to_stage
    }, ckpt_path)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": model.state_dict(),
            "class_names": class_names,
            "class_to_idx": class_to_idx,
            "class_idx_to_stage": class_idx_to_stage
        }, model_save_path)
        print("  --> New best model saved.")

# ---------------- Plot training curves ----------------
epochs = [h["epoch"] for h in history]
train_losses = [h["train_loss"] for h in history]
val_losses = [h["val_loss"] for h in history]
val_accs = [h["val_acc"] for h in history]

plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(epochs, train_losses, label="train_loss")
plt.plot(epochs, val_losses, label="val_loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.title("Loss")

plt.subplot(1,2,2)
plt.plot(epochs, val_accs, label="val_acc")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy"); plt.legend(); plt.title("Val Accuracy")

plt.show()

print("Training finished. Best validation acc:", best_val_acc)
print("Best model saved at:", model_save_path)
print("Checkpoints saved in:", checkpoint_dir)
print("CSV log:", log_csv)
